# Lab Day 1 — Notebook (PSEUDO-CODE)

> **Đây là khung pseudo-code.** Mọi ô có `# TODO` là phần bạn phải tự viết. Hãy giữ cấu trúc mục (Part 0–4) và
> viết dự đoán/nhận xét bằng chữ của bạn. Đọc `README.md`, `GUIDE.md`, `RUBRIC.md` trước.
>
> **Quy trình làm bài**
> 1. `git pull` repo, tạo thư mục nộp: copy cả thư mục `code/` vào `submission_<MSSV>/code/`.
> 2. Mở `submission_<MSSV>/code/lab.ipynb` và làm việc **tại đó** (nên `REPO_ROOT = "../.."` đúng và ảnh/kết quả ghi vào `submission_<MSSV>/figures`, `results`).
> 3. Ở Colab: chạy ô clone repo bên dưới, rồi làm tương tự (đặt `REPO_ROOT` cho đúng).
> 4. Cuối cùng: `Restart & Run All` phải chạy hết không lỗi và giữ nguyên output.

In [ ]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
import numpy as np, torch
import torch.nn.functional as F

REPO_ROOT = "../.."
OUT_DIR   = ".."
# !git clone https://github.com/VinUni-AI20k/K4-Track4-Day1-Neuralnetwork.git
# REPO_ROOT = "/content/K4-Track4-Day1-Neuralnetwork"

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}, Device: {device}")
if device == "cuda": print(torch.cuda.get_device_name(0))

os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx


## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [ ]:
# Split data if not already done
subprocess.run([sys.executable, f"{REPO_ROOT}/scripts/split_data.py"], check=False)
data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")
print(f"X_tr mean (first 10): {data['X_tr'][:, :10].mean(dim=0)}")
print(f"X_tr std (first 10): {data['X_tr'][:, :10].std(dim=0)}")


## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [ ]:
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
assert count_params(model) == EXPECTED_PARAMS[(256, 128)]
print(f"Params: {count_params(model)}")

dummy_x = torch.randn(8, 54).to(device)
logits = model(dummy_x)
print(f"Logits shape: {logits.shape}")
assert logits.shape == (8, 7)

step0_res = evaluate(model, data["X_val"], data["y_val"], "ce")
print(f"Step 0 val loss: {step0_res['loss']:.4f} (Expected ~1.946)")

# Overfit 20 samples
overfit_x = data["X_tr"][:20]
overfit_y = data["y_tr"][:20]
opt = torch.optim.SGD(model.parameters(), lr=0.1)
losses = []
for _ in range(500):
    opt.zero_grad()
    l = F.cross_entropy(model(overfit_x), overfit_y)
    l.backward()
    opt.step()
    losses.append(l.item())

import matplotlib.pyplot as plt
plt.plot(losses)
plt.title("Overfitting 20 samples")
plt.show()

# Grad norm
opt.zero_grad()
F.cross_entropy(model(overfit_x), overfit_y).backward()
for name, p in model.named_parameters():
    if p.grad is not None:
        print(f"{name} grad norm: {p.grad.norm().item():.6f}")
        assert p.grad.norm().item() > 0


**Nhận xét Part 1 (tự viết):** loss bước 0 đo được = ___, so với ln 7 = 1.946 vì sao ...; quá khớp 20 mẫu: ...

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [ ]:
lr_candidates = [0.001, 0.01, 0.1]
lr_results = []
for lr in lr_candidates:
    cfg = {**DEFAULT_CFG, "lr": lr, "epochs": 3, "exp_id": f"find_lr_{lr}"}
    res = run_experiment(cfg, data)
    lr_results.append((lr, res["summary"]["val_macro_f1"]))
print("LR Search:", lr_results)

best_lr = max(lr_results, key=lambda x: x[1])[0]
print(f"Selected LR: {best_lr}")

cfg = {**DEFAULT_CFG, "lr": best_lr}
baseline_results = []
for s in [1, 2, 3]:
    exp_cfg = {**cfg, "seed": s, "exp_id": f"base-s{s}"}
    res = run_experiment(exp_cfg, data)
    save_result(res, f"{OUT_DIR}/results")
    plot_run(res, f"{OUT_DIR}/figures/{exp_cfg['exp_id']}.png")
    baseline_results.append(res)
    
val_f1s = [r["summary"]["val_macro_f1"] for r in baseline_results]
val_accs = [r["summary"]["val_acc"] for r in baseline_results]
print(f"Baseline F1: {np.mean(val_f1s):.4f} ± {np.std(val_f1s):.4f}")
print(f"Baseline Acc: {np.mean(val_accs):.4f} ± {np.std(val_accs):.4f}")


**Nhận xét baseline (tự viết):** hình dạng đường cong (còn giảm? bắt đầu quá khớp?), val_acc so với mốc "đoán đa số", độ nhiễu giữa các seed ...

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Thí nghiệm `<exp_id>` — chủ đề `<nhóm>`
**Dự đoán trước (viết TRƯỚC khi chạy):** ...

In [ ]:
opts = ["adam", "adamw"]
exp_results = []
for opt in opts:
    exp_cfg = {**cfg, "optimizer": opt, "lr": 0.001, "exp_id": f"opt-{opt}", "group": "optimizer", "description": f"Optimizer {opt}"}
    res = run_experiment(exp_cfg, data)
    save_result(res, f"{OUT_DIR}/results")
    plot_run(res, f"{OUT_DIR}/figures/{exp_cfg['exp_id']}.png")
    exp_results.append(res)
    print(f"{opt} - Best Val F1: {res['summary']['val_macro_f1']:.4f}")


**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

In [ ]:
plot_compare(baseline_results[:1] + exp_results, "val_loss", f"{OUT_DIR}/figures/compare_optimizer.png", "Optimizer Comparison")


## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [ ]:
best_exp_res = max(baseline_results + exp_results, key=lambda x: x["summary"]["val_macro_f1"])
cfg_final = best_exp_res["cfg"]
print(f"Final Config chosen: {cfg_final['exp_id']}")

final_eval(cfg_final, best_exp_res, data, f"{OUT_DIR}/predictions_eval.csv")
subprocess.run([sys.executable, f"{REPO_ROOT}/scripts/evaluate.py", "--pred", f"{OUT_DIR}/predictions_eval.csv", "--out", f"{OUT_DIR}/eval_result.json"])

with open(f"{OUT_DIR}/eval_result.json", "r") as f:
    eval_res = json.load(f)
print("Eval Result:", eval_res["accuracy"], eval_res["macro_f1"])


In [ ]:
import pandas as pd
print("F1 by class:", eval_res.get("f1_by_class"))
cm = eval_res.get("confusion_matrix")
if cm:
    df_cm = pd.DataFrame(cm)
    print("Confusion Matrix:")
    print(df_cm)


**Phân tích lỗi (tự viết):** ...

In [ ]:
results = load_results(f"{OUT_DIR}/results")
rows = []
for r in results:
    if r["cfg"]["exp_id"] == cfg_final["exp_id"]:
        rows.append(to_row(r, eval_res))
    else:
        rows.append(to_row(r))
write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
print("Finished writing to Excel!")
